# Fase 7: Validacao e Perfis de Clusters

## Objetivo
1. Comparar metricas de todos os metodos
2. Criar perfis de clusters (radar charts)
3. Cross-validation com variavel Correto

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

OUTPUT_DIR = '/home/antonio/Documentos/Machine Learning/atividade_3'
IMG_DIR = f'{OUTPUT_DIR}/imgs'
numeric_cols = ['AL', 'ACD', 'WTW', 'K1', 'K2']

# Carregar dados
X_std = pd.read_pickle(f'{OUTPUT_DIR}/X_standard.pkl')
df = pd.read_pickle(f'{OUTPUT_DIR}/df_clean.pkl')

In [ ]:
# Carregar labels
kmeans_labels = np.load(f'{OUTPUT_DIR}/kmeans_labels.npy')
dbscan_labels = np.load(f'{OUTPUT_DIR}/dbscan_labels.npy')
hc_labels = np.load(f'{OUTPUT_DIR}/hierarchical_labels.npy')

In [ ]:
# Funcao para calcular metricas
def compute_metrics(X, labels, method_name):
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    
    mask = labels != -1
    if mask.sum() < n_clusters or n_clusters < 2:
        return {'Method': method_name, 'N_clusters': n_clusters, 'Silhouette': np.nan, 
                'Calinski-Harabasz': np.nan, 'Davies-Bouldin': np.nan}
    
    X_filter = X[mask] if mask.sum() < len(X) else X
    labels_filter = labels[mask] if mask.sum() < len(X) else labels
    
    sil = silhouette_score(X_filter, labels_filter)
    ch = calinski_harabasz_score(X_filter, labels_filter)
    db = davies_bouldin_score(X_filter, labels_filter)
    
    return {'Method': method_name, 'N_clusters': n_clusters, 
            'Silhouette': sil, 'CH': ch, 'DB': db}

In [ ]:
# Calcular metricas
metrics = []
metrics.append(compute_metrics(X_std.values, kmeans_labels, 'K-Means (K=2)'))
metrics.append(compute_metrics(X_std.values, dbscan_labels, 'DBSCAN'))
metrics.append(compute_metrics(X_std.values, hc_labels, 'Hierarchical (Ward)'))

results_df = pd.DataFrame(metrics)
results_df['Silhouette'] = results_df['Silhouette'].round(4)
results_df['CH'] = results_df['CH'].round(2)
results_df['DB'] = results_df['DB'].round(4)
print('=== Validacao de Metodos ===')
print(results_df.to_string(index=False))

results_df.to_csv(f'{OUTPUT_DIR}/validation_metrics.csv', index=False)

In [ ]:
# K-Means cluster profiles
df_kmeans = X_std.copy()
df_kmeans['Cluster'] = kmeans_labels
profile_kmeans = df_kmeans.groupby('Cluster').mean()
print('\n=== K-Means Cluster Profiles (Standardized) ===')
print(profile_kmeans.round(3))

In [ ]:
# Radar chart
cluster_0 = profile_kmeans.loc[0].values
cluster_1 = profile_kmeans.loc[1].values

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
n_vars = len(numeric_cols)
angles = np.linspace(0, 2 * np.pi, n_vars, endpoint=False).tolist()
angles += angles[:1]

ax.plot(angles, cluster_0.tolist() + [cluster_0[0]], 'o-', linewidth=2, label='Cluster 0', color='blue')
ax.fill(angles, cluster_0.tolist() + [cluster_0[0]], alpha=0.25, color='blue')
ax.plot(angles, cluster_1.tolist() + [cluster_1[0]], 'o-', linewidth=2, label='Cluster 1', color='red')
ax.fill(angles, cluster_1.tolist() + [cluster_1[0]], alpha=0.25, color='red')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(numeric_cols, fontsize=12)
ax.set_title('K-Means: Cluster Profiles (Radar)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/radar_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross with Correto
df['KMeans_Cluster'] = kmeans_labels
cross = pd.crosstab(df['Correto'], df['KMeans_Cluster'])
print('\n=== Cross: Correto vs K-Means ===')
print(cross)

chi2, p, dof, expected = chi2_contingency(cross)
print(f'\nChi2={chi2:.2f}, p-value={p:.4f}')